# TP1 : Systèmes de recommandation

**Objectif :** Implémenter un système de recommandation de films sur le jeu de données MovieLens (`ml-latest-small`).

On importe les librairies nécessaires : `pandas` pour la manipulation des données tabulaires et `numpy` pour le calcul matriciel.

In [1]:
import pandas as pd
import numpy as np

## Question 1.a

Rendez-vous sur le site de MovieLens et téléchargez le fichier `ml-latest-small.zip`. Le dataset est déjà présent dans le dossier `ml-latest-small/` (fichiers `ratings.csv`, `movies.csv`, `tags.csv`, `links.csv`).

On charge le fichier `ratings.csv` et on inspecte les identifiants uniques d'utilisateurs et de films.

In [2]:
file_path = "ml-latest-small/ratings.csv"
df = pd.read_csv(file_path)

u_users = df['userId'].unique()
u_movies = df['movieId'].unique()

print(df.head())
print(f"Number of unique users: {len(u_users)}")
print(f"Number of unique movies: {len(u_movies)}")

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
Number of unique users: 610
Number of unique movies: 9724


## Question 1.b

Écrire une fonction `load_data()` qui :
- Charge les données du fichier `ratings.csv`.
- Prend la liste des utilisateurs et films uniques.
- Associe un identifiant unique (commençant à 0) à chaque utilisateur et objet.
- Initialise la matrice d'utilité avec des zéros (numpy), puis la remplit avec les notes.

La fonction retourne la matrice d'utilité.

In [3]:
def load_data(file_path="ml-latest-small/ratings.csv"):
    df = pd.read_csv(file_path)

    u_users = df['userId'].unique()
    u_movies = df['movieId'].unique()

    users_obj = {user_id: idx for idx, user_id in enumerate(u_users)}
    movies_obj = {movie_id: idx for idx, movie_id in enumerate(u_movies)}

    utility_matrix = np.zeros((len(u_users), len(u_movies)))
    for row in df.itertuples():
        user_idx = users_obj[row.userId]
        movie_idx = movies_obj[row.movieId]
        utility_matrix[user_idx, movie_idx] = row.rating

    return utility_matrix, users_obj, movies_obj

On construit la matrice d'utilité et on vérifie sa forme ainsi que sa sparsité (proportion de notes connues).

In [4]:
utility_matrix, users_obj, movies_obj = load_data()
print("Utility Matrix:", utility_matrix.shape)
print(f"Sparsity: {np.sum(utility_matrix > 0) / utility_matrix.size * 100:.2f}%")

Utility Matrix: (610, 9724)
Sparsity: 1.70%


## Question 1.c

Pour cet exercice, la mesure de similarité est la similarité cosinus. Écrire une fonction `get_similarities(utility)` retournant :
- la matrice de tous les produits vectoriels d'utilisateurs,
- la matrice de tous les produits vectoriels de films,
- un vecteur avec l'inverse des normes de tous les utilisateurs,
- un vecteur avec l'inverse des normes de tous les objets.

On évite les boucles et on utilise uniquement des opérations matricielles.

In [5]:
def get_similarities(utility):
    user_sim = utility @ utility.T
    item_sim = utility.T @ utility

    user_norms = np.linalg.norm(utility, axis=1)
    item_norms = np.linalg.norm(utility, axis=0)

    # avoid division by zero for users/items with no ratings
    inv_user_norm = np.divide(1.0, user_norms, out=np.zeros_like(user_norms), where=user_norms != 0)
    inv_item_norm = np.divide(1.0, item_norms, out=np.zeros_like(item_norms), where=item_norms != 0)

    return user_sim, item_sim, inv_user_norm, inv_item_norm

In [6]:
user_sim, item_sim, inv_user_norm, inv_item_norm = get_similarities(utility_matrix)
print("user_sim:", user_sim.shape)
print("item_sim:", item_sim.shape)
print("inv_user_norm:", inv_user_norm.shape)
print("inv_item_norm:", inv_item_norm.shape)

user_sim: (610, 610)
item_sim: (9724, 9724)
inv_user_norm: (610,)
inv_item_norm: (9724,)


## Question 1.d

Écrire deux fonctions `rank_users(id_user, similarities, inv_norm)` et `rank_items(id_movie, similarities, inv_norm)` qui retournent le classement de tous les utilisateurs (ou films) selon leur score de similarité cosinus. Le format de sortie est une liste de couples `(user, similarité)` ou `(item, similarité)`, triée par similarité décroissante.

In [7]:
def _rank(id_, similarities, inv_norm):
    # cosine similarity of id_ with every other index, using precomputed dot products and inverse norms
    cos_sim = similarities[id_] * inv_norm[id_] * inv_norm
    order = np.argsort(-cos_sim)
    return [(idx, cos_sim[idx]) for idx in order]

def rank_users(id_user, similarities, inv_norm):
    return _rank(id_user, similarities, inv_norm)

def rank_items(id_movie, similarities, inv_norm):
    return _rank(id_movie, similarities, inv_norm)

In [8]:
# sanity check: the user itself should rank first
print(rank_users(0, user_sim, inv_user_norm)[:5])
print(rank_items(0, item_sim, inv_item_norm)[:5])

[(np.int64(0), np.float64(1.0)), (np.int64(265), np.float64(0.35740770960327417)), (np.int64(312), np.float64(0.3515615184909569)), (np.int64(367), np.float64(0.34512705158353046)), (np.int64(56), np.float64(0.34503427880758364))]
[(np.int64(0), np.float64(0.9999999999999999)), (np.int64(735), np.float64(0.5726012603197153)), (np.int64(26), np.float64(0.5656368040861565)), (np.int64(42), np.float64(0.5642616935276659)), (np.int64(15), np.float64(0.5573881705799365))]


## Question 1.e

Écrire la fonction `recommend_user_based_item(id_user, id_item, utility, similarities, inv_norm, k=10)` qui prédit le score pour un utilisateur et un objet donnés, avec une approche basée sur l'utilisateur (moyenne pondérée des notes des `k` utilisateurs les plus proches ayant noté l'objet).

In [9]:
def recommend_user_based_item(id_user, id_item, utility, similarities, inv_norm, k=10):
    ranked = rank_users(id_user, similarities, inv_norm)

    num, den = 0.0, 0.0
    count = 0
    for neighbor, sim in ranked:
        if neighbor == id_user:
            continue
        rating = utility[neighbor, id_item]
        if rating == 0:
            continue
        num += sim * rating
        den += abs(sim)
        count += 1
        if count == k:
            break

    if den == 0:
        return 0.0
    return num / den

In [10]:
print("Predicted score:", recommend_user_based_item(0, 1, utility_matrix, user_sim, inv_user_norm, k=10))

Predicted score: 2.951053817089232


## Question 1.f

De même, écrire la fonction `recommend_item_based_item(id_user, id_item, utility, similarities, inv_norm, k=10)` qui prédit le score avec une approche basée sur l'objet (moyenne pondérée des notes données par l'utilisateur aux `k` objets les plus proches de l'objet cible).

In [11]:
def recommend_item_based_item(id_user, id_item, utility, similarities, inv_norm, k=10):
    ranked = rank_items(id_item, similarities, inv_norm)

    num, den = 0.0, 0.0
    count = 0
    for neighbor, sim in ranked:
        if neighbor == id_item:
            continue
        rating = utility[id_user, neighbor]
        if rating == 0:
            continue
        num += sim * rating
        den += abs(sim)
        count += 1
        if count == k:
            break

    if den == 0:
        return 0.0
    return num / den

In [12]:
print("Predicted score:", recommend_item_based_item(0, 1, utility_matrix, item_sim, inv_item_norm, k=10))

Predicted score: 4.304958360064558


## Question 1.g

On modifie `load_data()` pour séparer le dataset en un jeu d'entraînement et de test. On mélange les lignes (`df.sample(frac=1)`), puis on met les 80% premières lignes dans le train et les 20% restants dans le test. Le train sert à construire la matrice d'utilité. Le test est transformé en une liste de `(userId, movieId, rating)`. La fonction retourne maintenant un couple : la matrice d'utilité et la liste des tests.

In [13]:
def load_data(file_path="ml-latest-small/ratings.csv", test_frac=0.2, seed=42, center=False):
    df = pd.read_csv(file_path)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

    u_users = df['userId'].unique()
    u_movies = df['movieId'].unique()
    users_obj = {user_id: idx for idx, user_id in enumerate(u_users)}
    movies_obj = {movie_id: idx for idx, movie_id in enumerate(u_movies)}

    n_test = int(len(df) * test_frac)
    train_df = df.iloc[:-n_test] if n_test > 0 else df
    test_df = df.iloc[-n_test:] if n_test > 0 else df.iloc[0:0]

    utility_matrix = np.zeros((len(u_users), len(u_movies)))
    for row in train_df.itertuples():
        utility_matrix[users_obj[row.userId], movies_obj[row.movieId]] = row.rating

    train_mean = 0.0
    if center:
        # Question 1.o: remove the mean of known ratings on the training set
        mask = utility_matrix != 0
        train_mean = utility_matrix[mask].mean()
        utility_matrix[mask] -= train_mean

    test_set = [
        (users_obj[row.userId], movies_obj[row.movieId], row.rating)
        for row in test_df.itertuples()
        if row.userId in users_obj and row.movieId in movies_obj
    ]

    return utility_matrix, test_set, users_obj, movies_obj, train_mean

In [14]:
utility_matrix, test_set, users_obj, movies_obj, train_mean = load_data()
user_sim, item_sim, inv_user_norm, inv_item_norm = get_similarities(utility_matrix)
print("Utility Matrix:", utility_matrix.shape)
print("Test set size:", len(test_set))

Utility Matrix: (610, 9724)
Test set size: 20167


## Question 1.h

Implémentez la fonction `get_rmse_user_based(test_set, utility, similarities, inv_norm, k=10)` qui calcule la RMSE pour l'approche basée sur l'utilisateur. De même, implémentez `get_rmse_item_based(test_set, utility, similarities, inv_norm, k=10)` pour l'approche basée sur l'objet. On compare ensuite les résultats.

In [15]:
def get_rmse_user_based(test_set, utility, similarities, inv_norm, k=10):
    errors = []
    for id_user, id_item, rating in test_set:
        pred = recommend_user_based_item(id_user, id_item, utility, similarities, inv_norm, k=k)
        errors.append((pred - rating) ** 2)
    return np.sqrt(np.mean(errors))

def get_rmse_item_based(test_set, utility, similarities, inv_norm, k=10):
    errors = []
    for id_user, id_item, rating in test_set:
        pred = recommend_item_based_item(id_user, id_item, utility, similarities, inv_norm, k=k)
        errors.append((pred - rating) ** 2)
    return np.sqrt(np.mean(errors))

In [16]:
# Note: this loops the full test set (~20k pairs) and recomputes a full ranking each time, so it may take a while.
rmse_user = get_rmse_user_based(test_set, utility_matrix, user_sim, inv_user_norm, k=10)
rmse_item = get_rmse_item_based(test_set, utility_matrix, item_sim, inv_item_norm, k=10)
print(f"RMSE user-based: {rmse_user:.4f}")
print(f"RMSE item-based: {rmse_item:.4f}")

RMSE user-based: 1.1884
RMSE item-based: 1.1054


## Question 1.i

Écrire la méthode `get_ndcg_at_k_user_based(test_set, utility, similarities, inv_norm, k=10, k_ndcg=10)` qui calcule le NDCG@K avec une approche basée sur les utilisateurs. On applique l'approche naïve consistant à parcourir tous les utilisateurs du test set, prédire un score pour chaque objet, et classer en fonction de ces scores.

Cette méthode naïve étant très longue à calculer, on peut accélérer les calculs en précalculant pour chaque utilisateur les utilisateurs les plus proches.

In [17]:
def _dcg(relevances):
    relevances = np.asarray(relevances, dtype=float)
    discounts = np.log2(np.arange(2, len(relevances) + 2))
    return np.sum(relevances / discounts)

def _ndcg(predicted_order_relevances, ideal_relevances):
    dcg = _dcg(predicted_order_relevances)
    idcg = _dcg(ideal_relevances)
    if idcg == 0:
        return 0.0
    return dcg / idcg

def get_ndcg_at_k_user_based(test_set, utility, similarities, inv_norm, k=10, k_ndcg=10):
    from collections import defaultdict
    truth_by_user = defaultdict(dict)
    for id_user, id_item, rating in test_set:
        truth_by_user[id_user][id_item] = rating

    scores = []
    for id_user, items_ratings in truth_by_user.items():
        # precompute the k nearest neighbors once per user
        ranked_neighbors = rank_users(id_user, similarities, inv_norm)[1:k + 1]

        preds = []
        for id_item, true_rating in items_ratings.items():
            num, den = 0.0, 0.0
            for neighbor, sim in ranked_neighbors:
                r = utility[neighbor, id_item]
                if r == 0:
                    continue
                num += sim * r
                den += abs(sim)
            pred = num / den if den != 0 else 0.0
            preds.append((pred, true_rating))

        preds_sorted = sorted(preds, key=lambda x: -x[0])
        predicted_relevances = [r for _, r in preds_sorted[:k_ndcg]]
        ideal_relevances = sorted((r for _, r in preds), reverse=True)[:k_ndcg]
        scores.append(_ndcg(predicted_relevances, ideal_relevances))

    return np.mean(scores)

In [18]:
ndcg_user = get_ndcg_at_k_user_based(test_set, utility_matrix, user_sim, inv_user_norm, k=10, k_ndcg=10)
print(f"NDCG@10 user-based: {ndcg_user:.4f}")

NDCG@10 user-based: 0.8923


## Question 1.j

De même, écrire la méthode `get_ndcg_at_k_item_based(test_set, utility, similarities, inv_norm, k=10, k_ndcg=10)` qui calcule le NDCG@K avec une approche basée sur les objets.

In [19]:
def get_ndcg_at_k_item_based(test_set, utility, similarities, inv_norm, k=10, k_ndcg=10):
    from collections import defaultdict
    truth_by_user = defaultdict(dict)
    for id_user, id_item, rating in test_set:
        truth_by_user[id_user][id_item] = rating

    scores = []
    for id_user, items_ratings in truth_by_user.items():
        preds = []
        for id_item, true_rating in items_ratings.items():
            ranked_neighbors = rank_items(id_item, similarities, inv_norm)[1:k + 1]
            num, den = 0.0, 0.0
            for neighbor, sim in ranked_neighbors:
                r = utility[id_user, neighbor]
                if r == 0:
                    continue
                num += sim * r
                den += abs(sim)
            pred = num / den if den != 0 else 0.0
            preds.append((pred, true_rating))

        preds_sorted = sorted(preds, key=lambda x: -x[0])
        predicted_relevances = [r for _, r in preds_sorted[:k_ndcg]]
        ideal_relevances = sorted((r for _, r in preds), reverse=True)[:k_ndcg]
        scores.append(_ndcg(predicted_relevances, ideal_relevances))

    return np.mean(scores)

In [20]:
ndcg_item = get_ndcg_at_k_item_based(test_set, utility_matrix, item_sim, inv_item_norm, k=10, k_ndcg=10)
print(f"NDCG@10 item-based: {ndcg_item:.4f}")

NDCG@10 item-based: 0.8958


## Question 1.k

On implémente l'approche basée sur un modèle à l'aide de l'algorithme SVD. On écrit la fonction `get_rmse_svd(test_set, utility, N=100)` qui calcule la RMSE de l'approche SVD. Dans Numpy, pour restreindre la SVD au rang `N` souhaité, on tronque manuellement les matrices `U`, `S` et `V`.

In [21]:
def svd_predict(utility, N=100):
    U, S, Vt = np.linalg.svd(utility, full_matrices=False)
    U_k = U[:, :N]
    S_k = S[:N]
    Vt_k = Vt[:N, :]
    return U_k @ np.diag(S_k) @ Vt_k

def get_rmse_svd(test_set, utility, N=100):
    reconstructed = svd_predict(utility, N=N)
    errors = [(reconstructed[u, i] - r) ** 2 for u, i, r in test_set]
    return np.sqrt(np.mean(errors))

In [22]:
rmse_svd = get_rmse_svd(test_set, utility_matrix, N=100)
print(f"RMSE SVD (N=100): {rmse_svd:.4f}")

RMSE SVD (N=100): 3.3712


## Question 1.l

On fait varier le rang `k` (la troncature `N` de la SVD) pour trouver la valeur qui minimise la RMSE.

In [23]:
ranks = [5, 10, 20, 50, 100, 150, 200, 300]
rmse_by_rank = {N: get_rmse_svd(test_set, utility_matrix, N=N) for N in ranks}

for N, rmse in rmse_by_rank.items():
    print(f"N={N:>4}  RMSE={rmse:.4f}")

best_N = min(rmse_by_rank, key=rmse_by_rank.get)
print(f"\nBest rank: N={best_N} (RMSE={rmse_by_rank[best_N]:.4f})")

N=   5  RMSE=2.9723
N=  10  RMSE=2.9572
N=  20  RMSE=3.0183
N=  50  RMSE=3.1827
N= 100  RMSE=3.3712
N= 150  RMSE=3.4739
N= 200  RMSE=3.5352
N= 300  RMSE=3.6015

Best rank: N=10 (RMSE=2.9572)


## Question 1.m

De même, on écrit la fonction `get_ndcg_svd(test_set, utility, k_ndcg=100, N=100)` qui calcule le NDCG dans le cadre de l'approche SVD. La vitesse d'exécution est ici bien plus rapide qu'avec les approches précédentes, puisque la matrice reconstruite est précalculée une seule fois.

In [24]:
def get_ndcg_svd(test_set, utility, k_ndcg=100, N=100):
    from collections import defaultdict
    reconstructed = svd_predict(utility, N=N)

    truth_by_user = defaultdict(dict)
    for id_user, id_item, rating in test_set:
        truth_by_user[id_user][id_item] = rating

    scores = []
    for id_user, items_ratings in truth_by_user.items():
        preds = [(reconstructed[id_user, id_item], rating) for id_item, rating in items_ratings.items()]
        preds_sorted = sorted(preds, key=lambda x: -x[0])
        predicted_relevances = [r for _, r in preds_sorted[:k_ndcg]]
        ideal_relevances = sorted((r for _, r in preds), reverse=True)[:k_ndcg]
        scores.append(_ndcg(predicted_relevances, ideal_relevances))

    return np.mean(scores)

In [25]:
ndcg_svd = get_ndcg_svd(test_set, utility_matrix, k_ndcg=10, N=100)
print(f"NDCG@10 SVD (N=100): {ndcg_svd:.4f}")

NDCG@10 SVD (N=100): 0.8875


## Question 1.n

On affiche les scores de prédiction de la SVD pour quelques paires (utilisateur, objet) du test set.

In [26]:
reconstructed = svd_predict(utility_matrix, N=best_N)
for id_user, id_item, rating in test_set[:10]:
    print(f"user={id_user:<4} item={id_item:<5} true={rating:<4} pred={reconstructed[id_user, id_item]:.3f}")

user=45   item=5736  true=4.0  pred=0.234
user=8    item=2072  true=4.0  pred=-0.037
user=335  item=92    true=5.0  pred=0.034
user=88   item=8928  true=4.0  pred=0.000
user=449  item=729   true=4.0  pred=0.926
user=99   item=2380  true=1.0  pred=0.258
user=39   item=54    true=4.0  pred=0.955
user=1    item=2887  true=2.5  pred=0.094
user=36   item=4261  true=2.5  pred=-0.272
user=483  item=1009  true=4.0  pred=0.507


**Remarque :** de nombreuses paires étaient à 0 dans la matrice d'entraînement (note inconnue). La SVD, en reconstruisant la matrice, a tendance à prédire des scores proches de 0 (ou en tout cas très bas comparés aux vraies notes qui vont de 0.5 à 5), car elle apprend à approximer une matrice majoritairement composée de zéros. Les prédictions sont donc systématiquement biaisées vers le bas.

## Question 1.o

Pour résoudre ce problème, on enlève la moyenne des notes du jeu d'entraînement (calculée uniquement sur les valeurs connues) avant d'appliquer la SVD. On modifie `load_data` en conséquence (paramètre `center=True`), puis on ajoute cette moyenne aux prédictions reconstruites. On compare enfin les performances avec et sans centrage.

In [27]:
utility_centered, test_set_centered, users_obj_c, movies_obj_c, train_mean = load_data(center=True)

def get_rmse_svd_centered(test_set, utility, mean, N=100):
    reconstructed = svd_predict(utility, N=N) + mean
    errors = [(reconstructed[u, i] - r) ** 2 for u, i, r in test_set]
    return np.sqrt(np.mean(errors))

def get_ndcg_svd_centered(test_set, utility, mean, k_ndcg=100, N=100):
    from collections import defaultdict
    reconstructed = svd_predict(utility, N=N) + mean

    truth_by_user = defaultdict(dict)
    for id_user, id_item, rating in test_set:
        truth_by_user[id_user][id_item] = rating

    scores = []
    for id_user, items_ratings in truth_by_user.items():
        preds = [(reconstructed[id_user, id_item], rating) for id_item, rating in items_ratings.items()]
        preds_sorted = sorted(preds, key=lambda x: -x[0])
        predicted_relevances = [r for _, r in preds_sorted[:k_ndcg]]
        ideal_relevances = sorted((r for _, r in preds), reverse=True)[:k_ndcg]
        scores.append(_ndcg(predicted_relevances, ideal_relevances))

    return np.mean(scores)

rmse_svd_centered = get_rmse_svd_centered(test_set_centered, utility_centered, train_mean, N=best_N)
ndcg_svd_centered = get_ndcg_svd_centered(test_set_centered, utility_centered, train_mean, k_ndcg=10, N=best_N)

print(f"RMSE SVD without centering (N={best_N}): {rmse_by_rank[best_N]:.4f}")
print(f"RMSE SVD with mean-centering (N={best_N}):  {rmse_svd_centered:.4f}")
print(f"NDCG@10 SVD without centering: {ndcg_svd:.4f}")
print(f"NDCG@10 SVD with mean-centering:  {ndcg_svd_centered:.4f}")

RMSE SVD without centering (N=10): 2.9572
RMSE SVD with mean-centering (N=10):  0.9991
NDCG@10 SVD without centering: 0.8875
NDCG@10 SVD with mean-centering:  0.9032


En retirant la moyenne globale des notes connues avant la SVD, les prédictions ne sont plus tirées vers 0 : la matrice décomposée est centrée, donc les valeurs manquantes (remplacées par 0 après centrage) représentent maintenant la note moyenne plutôt qu'une absence de note. Après avoir rajouté la moyenne aux prédictions, la RMSE diminue généralement par rapport à l'approche sans centrage, ce qui confirme que ce biais expliquait une bonne partie de l'erreur observée à la question 1.n.